The Plack Low can not fit with linear regression.  
It is a non liear process. The problem can not be transform into format of y = WX.  
I tried with logarithm. The equation is still a non-linear process.
So it might be solved with other modle, but not with linear-regression.

As well, for very larg value. Must pay attention to if the scale of varient exceed.

In [130]:
import numpy as np
import torch

In [131]:
def planck(lam, T):
    h = 6.626e-34       # Planck constant
    c = 3.0e8           # speed of light
    k = 1.381e-23       # Boltzmann constant

    numerator = 2*h*c**2

    denominator = (
        lam**5 *
        (torch.exp(h*c/(lam*k*T))-1)
    )

    return torch.log(numerator / denominator)

In [132]:
lam = torch.tensor([
    400e-9,
    500e-9,
    600e-9,
    700e-9
], dtype=torch.float64)

# measured spectral density
T_val = torch.tensor([
    5000,
    5000,
    5000,
    5000
], dtype=torch.float64)
'''
B_measure = torch.tensor([
    1.2e12,
    2.8e12,
    3.5e12,
    2.9e12
], dtype=torch.float64)
'''
B_measure = planck(lam, T_val)

In [133]:
T = torch.tensor([
    2000,
    2000,
    2000,
    2000
], dtype=torch.float64, requires_grad=True)

B_predict_log = planck(lam, T)

loss = torch.mean((B_predict_log - B_measure) ** 2)

In [134]:
loss.backward()
print(T.grad)

tensor([-0.0486, -0.0311, -0.0216, -0.0159], dtype=torch.float64)


In [135]:
lr = 1
for epoch in range(0,1000):
    B_predict_log = planck(lam,T)

    loss = torch.mean(
        (B_predict_log - B_measure)**2
    )

    loss.backward()

    with torch.no_grad():
        T -= lr*T.grad

    T.grad.zero_()

print(T)

tensor([2046.5778, 2030.2687, 2021.2121, 2015.6879], dtype=torch.float64,
       requires_grad=True)
